# I²C 통신


## 학습 목표

<!-- exercise-guidance:microprocessor/i2c/objective -->
같은 두 Byte도 부호와 센서 범위 설정에 따라 전혀 다른 물리량이 됩니다. 경계값부터 변환을 검증합니다.
<!-- /exercise-guidance:microprocessor/i2c/objective -->


## 실행 조건

<!-- exercise-guidance:microprocessor/i2c/execution -->
기본 실습: 제공한 byte·합성 센서 데이터. 실제 I²C 예제: Linux의 I²C 장치와 `smbus2` 필요 (`python -m pip install smbus2`).
<!-- /exercise-guidance:microprocessor/i2c/execution -->


## 두 바이트를 signed 값으로

<!-- exercise-guidance:microprocessor/i2c/activity-1 -->
### 시작 조건

바이트 쌍은 (0x40,0x00),(0xC0,0x00),(0x7F,0xFF),(0x80,0x00),(0xFF,0xFF)입니다. 앞 바이트가 high byte입니다.

### 진행 방법

1. (high<<8)|low로 16-bit unsigned 값을 만드세요.
2. 값이 2^15 이상이면 2^16을 빼 signed 값으로 바꾸세요.
3. 0x8000에서 raw>=0x8000과 raw>0x8000의 결과를 비교하세요.

### 남길 결과

- 5행 변환 표와 경계값 assert 2개.
<!-- /exercise-guidance:microprocessor/i2c/activity-1 -->


In [ ]:
pairs=[(0x40,0),(0xC0,0),(0x7F,0xFF),(0x80,0),(0xFF,0xFF)]
def signed16(high,low):
    value=(high<<8)|low
    return value-65536 if value>=32768 else value
print([signed16(*p) for p in pairs])
# TODO: unsigned·예상 값과 경계 assert를 추가하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/i2c/activity-1-record -->
**high:**

**low:**

**unsigned:**

**signed:**

**직접 예상:**

### 확인할 점

- signed 범위 -32768~32767 안에 들어가는가?
- 0x8000은 -32768로 해석되는가?
<!-- /exercise-guidance:microprocessor/i2c/activity-1-record -->


## Raw 값과 물리 단위

<!-- exercise-guidance:microprocessor/i2c/activity-2 -->
### 시작 조건

가속도 $\mathrm{raw}=\begin{bmatrix}16384&-8192&0\end{bmatrix}$, 감도는 16384 LSB/g와 8192 LSB/g 두 조건입니다. Gyro raw=1310에 131 또는 16.4 LSB/(°/s)를 적용합니다.

### 진행 방법

1. 각 감도 조건에서 가속도와 Angular Velocity를 계산하세요.
2. 같은 raw 값이라도 센서 설정이 바뀌면 물리량이 달라지는 이유를 적으세요.
3. [Week7-1_I2C_IMU_manual.ipynb](https://chorok-daddy.github.io/niil/courses/microprocessor/i2c/Week7-1_I2C_IMU_manual.preview.html)의 `MPU_Init` 안에서 `GYRO_CONFIG`에 쓰는 값과 Gyro raw를 나누는 부분을 읽으세요. 레지스터 비트·감도·단위로 나누어 설정이 맞는지 판단할 항목을 적으세요. 이 단계는 코드 읽기이며 장치 연결이나 파일 실행은 필요하지 않습니다.

### 남길 결과

- 가속도·Gyro 변환 표와 설정 확인 항목 3개.
<!-- /exercise-guidance:microprocessor/i2c/activity-2 -->


In [ ]:
acc_raw=[16384,-8192,0]
for scale in [16384.,8192.]:print('acc [g]',[v/scale for v in acc_raw])
print('gyro [deg/s]',1310/131,1310/16.4)
# TODO: 감도·단위 표와 원본 설정 대조 항목을 적으세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/i2c/activity-2-record -->
**raw:**

**설정 감도:**

**변환 값:**

**단위:**

### 확인할 점

- 이미 g 또는 °/s인 라이브러리 출력을 다시 raw처럼 나누지 않았는가?
- MPU6050과 MPU9250을 같은 센서·같은 변환 상수라고 가정하지 않았는가?
<!-- /exercise-guidance:microprocessor/i2c/activity-2-record -->


## 정지 센서 기록의 편차

<!-- exercise-guidance:microprocessor/i2c/activity-3 -->
### 시작 조건

`seed=7`로 [0,0,16384] LSB에 표준편차 120 LSB의 노이즈를 더한 100개 3축 표본을 생성합니다. 감도는 16384 LSB/g입니다.

### 진행 방법

1. 두 조건의 raw 배열을 감도 16384 LSB/g로 나누어 가속도 단위로 바꾸세요. 각 배열의 `mean(axis=0)`과 `std(axis=0)`으로 축별 평균·표준편차를 구하세요.
2. 각 행의 세 축 값에서 Euclidean Norm을 구해 표본별 가속도 크기를 계산하세요. 원래 기록과 X축 500 LSB Bias를 더한 기록에 같은 연산을 적용하세요.
3. 조건별 축 평균·축 표준편차·가속도 크기의 평균을 한 표에 적고 3축 시계열을 그리세요. X축 평균 변화와 표준편차 변화를 구분해 Offset과 Noise의 차이를 설명하세요.

### 남길 결과

- 두 조건 표와 3축 시계열 그림.
<!-- /exercise-guidance:microprocessor/i2c/activity-3 -->


In [ ]:
import numpy as np
rng=np.random.default_rng(7)
raw=np.array([0,0,16384])+rng.normal(0,120,size=(100,3))
biased=raw+np.array([500,0,0])
# TODO: g 변환과 통계·그래프를 작성하세요.


## 결과 기록·자가 점검

<!-- exercise-guidance:microprocessor/i2c/activity-3-record -->
**조건:**

**축별 평균 [g]:**

**축별 표준편차 [g]:**

**평균 벡터 크기 [g]:**

### 확인할 점

- 같은 노이즈 표본에 bias만 추가했는가?
- 평균과 표준편차를 서로 다른 지표로 읽었는가?
<!-- /exercise-guidance:microprocessor/i2c/activity-3-record -->
